# Colab Pipeline - IndoBERT POS Tagging (Pantun Gaul)

Notebook ini menjalankan pipeline end-to-end:
1. Train token-classification POS dengan IndoBERT (Lite/full bisa switch)
2. Evaluasi ke UD test set
3. Prediksi 100 pantun awal
4. Evaluasi mini-gold (jika file mini-gold tersedia)

Rekomendasi awal: mulai dari **IndoBERT Lite** untuk iterasi cepat, lalu bandingkan dengan IndoBERT full jika resource cukup.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import subprocess
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_PARENT = DRIVE_ROOT / 'POSTagPantun'
REPO_URL = 'https://github.com/NazirArifin/pantunify.git'
REPO_DIR = PROJECT_PARENT / 'pantunify'
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists, skip clone.')

%cd {REPO_DIR}
!git --no-pager status -sb


In [ ]:
!python -V
!pip -q install --upgrade pip
!pip -q install torch transformers datasets accelerate scikit-learn pandas


In [ ]:
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import confusion_matrix
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

# ===== PILIH MODEL =====
# Opsi Lite: indobenchmark/indobert-lite-base-p1
# Opsi Full: indobenchmark/indobert-base-p1
MODEL_NAME = 'indobenchmark/indobert-lite-base-p1'
# MODEL_NAME = 'indobenchmark/indobert-base-p1'

SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 6
LR = 3e-5
WEIGHT_DECAY = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = REPO_DIR / 'data' / 'article5'

# Simpan artefak DI LUAR repo clone supaya aman saat repo dihapus/update.
ARTIFACT_ROOT = PROJECT_PARENT / 'artifacts_indobert'
MODEL_ROOT = PROJECT_PARENT / 'models_indobert'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ALIAS = MODEL_NAME.replace('/', '__')
OUT_DIR = ARTIFACT_ROOT / MODEL_ALIAS
RUN_DIR = MODEL_ROOT / MODEL_ALIAS
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILES = [
    DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-train.conllu',
    DATA_DIR / 'id_csui-ud-train.conllu',
]
DEV_FILE = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-dev.conllu'
TEST_FILE = DATA_DIR / 'UD_Indonesian-GSD' / 'id_gsd-ud-test.conllu'
PANTUN_CSV = DATA_DIR / 'pantun_gaul.csv'

for p in [*TRAIN_FILES, DEV_FILE, TEST_FILE, PANTUN_CSV]:
    print(p, '->', 'OK' if p.exists() else 'MISSING')
print('MODEL_NAME =', MODEL_NAME)
print('MODEL DIR  =', RUN_DIR)
print('OUTPUT DIR =', OUT_DIR)


In [ ]:
def parse_conllu(path: Path) -> List[Dict[str, List[str]]]:
    sentences = []
    tokens, tags = [], []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append({'tokens': tokens, 'tags': tags})
                    tokens, tags = [], []
                continue
            if line.startswith('#'):
                continue
            cols = line.split('\t')
            if len(cols) < 4:
                continue
            tid, form, upos = cols[0], cols[1], cols[3]
            if '-' in tid or '.' in tid:
                continue
            if upos == '_':
                continue
            token = form.strip()
            if not token or token == '_':
                continue
            tokens.append(token)
            tags.append(upos)
    if tokens:
        sentences.append({'tokens': tokens, 'tags': tags})
    return sentences


train_sentences = []
for fp in TRAIN_FILES:
    train_sentences.extend(parse_conllu(fp))
dev_sentences = parse_conllu(DEV_FILE)
test_sentences = parse_conllu(TEST_FILE)

all_tags = sorted({tag for s in train_sentences + dev_sentences + test_sentences for tag in s['tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}

print('train:', len(train_sentences), 'dev:', len(dev_sentences), 'test:', len(test_sentences))
print('num tags:', len(all_tags), all_tags)

with (OUT_DIR / 'label_map.json').open('w', encoding='utf-8') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, ensure_ascii=False, indent=2)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(sentences: List[Dict[str, List[str]]]) -> Dataset:
    return Dataset.from_dict({
        'tokens': [s['tokens'] for s in sentences],
        'tags': [s['tags'] for s in sentences],
    })

train_ds = to_hf_dataset(train_sentences)
dev_ds = to_hf_dataset(dev_sentences)
test_ds = to_hf_dataset(test_sentences)

LABEL_ALL_SUBTOKENS = True

def tokenize_and_align_labels(batch):
    tok = tokenizer(
        batch['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
    )
    labels = []
    for i, tags in enumerate(batch['tags']):
        word_ids = tok.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word_id:
                label_ids.append(label2id[tags[wid]])
            else:
                label_ids.append(label2id[tags[wid]] if LABEL_ALL_SUBTOKENS else -100)
            prev_word_id = wid
        labels.append(label_ids)
    tok['labels'] = labels
    return tok

train_tok = train_ds.map(tokenize_and_align_labels, batched=True)
dev_tok = dev_ds.map(tokenize_and_align_labels, batched=True)
test_tok = test_ds.map(tokenize_and_align_labels, batched=True)

BEST_MODEL_DIR = RUN_DIR / 'best'
TRAIN_FROM_SCRATCH = True  # set False untuk load model existing tanpa retrain

if (not TRAIN_FROM_SCRATCH) and BEST_MODEL_DIR.exists():
    print('Loading existing model from', BEST_MODEL_DIR)
    model = AutoModelForTokenClassification.from_pretrained(str(BEST_MODEL_DIR))
    tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR))
else:
    print('Training from pretrained base model:', MODEL_NAME)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(all_tags),
        id2label=id2label,
        label2id=label2id,
    )

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

import inspect
import transformers
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

CRITICAL_TAGS = ['ADV', 'PART', 'NOUN', 'VERB']
critical_ids = [label2id[t] for t in CRITICAL_TAGS if t in label2id]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    # 1. Ratakan array (flatten) dan filter token padding/ignore index (-100)
    flat_labels = labels.flatten()
    flat_preds = preds.flatten()
    
    mask = flat_labels != -100
    actual_labels = flat_labels[mask]
    actual_preds = flat_preds[mask]
    
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        actual_labels, 
        actual_preds, 
        average='macro', 
        zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        actual_labels,
        actual_preds,
        average='weighted',
        zero_division=0,
    )
    acc = accuracy_score(actual_labels, actual_preds)

    if critical_ids:
        crit_mask = np.isin(actual_labels, np.array(critical_ids))
        crit_labels = actual_labels[crit_mask]
        crit_preds = actual_preds[crit_mask]
        if len(crit_labels) > 0:
            _, _, crit_f1_macro, _ = precision_recall_fscore_support(
                crit_labels,
                crit_preds,
                average='macro',
                zero_division=0,
            )
        else:
            crit_f1_macro = 0.0
    else:
        crit_f1_macro = 0.0
    
    return {
        'accuracy': acc,
        'precision': precision_macro,
        'recall': recall_macro,
        'f1': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'critical_f1_macro': crit_f1_macro,
    }

common_args = {
    'output_dir': str(RUN_DIR),
    'learning_rate': LR,
    'per_device_train_batch_size': BATCH_SIZE,
    'per_device_eval_batch_size': BATCH_SIZE,
    'num_train_epochs': EPOCHS,
    'weight_decay': WEIGHT_DECAY,
    'save_strategy': 'epoch',
    'logging_strategy': 'steps',
    'logging_steps': 100,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_weighted',
    'greater_is_better': True,
    'report_to': 'none',
    'seed': SEED,
}
sig = inspect.signature(TrainingArguments.__init__)
print('transformers version:', transformers.__version__)
if 'evaluation_strategy' in sig.parameters:
    args = TrainingArguments(evaluation_strategy='epoch', **common_args)
    print('TrainingArguments uses: evaluation_strategy')
elif 'eval_strategy' in sig.parameters:
    args = TrainingArguments(eval_strategy='epoch', **common_args)
    print('TrainingArguments uses: eval_strategy')
else:
    raise RuntimeError('Neither evaluation_strategy nor eval_strategy is supported in this transformers version.')

trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': train_tok,
    'eval_dataset': dev_tok,
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}
trainer_sig = inspect.signature(Trainer.__init__)
if 'tokenizer' in trainer_sig.parameters:
    trainer_kwargs['tokenizer'] = tokenizer
    print('Trainer uses: tokenizer')
elif 'processing_class' in trainer_sig.parameters:
    trainer_kwargs['processing_class'] = tokenizer
    print('Trainer uses: processing_class')
else:
    print('Trainer has neither tokenizer nor processing_class; continue without processor arg')
trainer = Trainer(**trainer_kwargs)

if TRAIN_FROM_SCRATCH or (not BEST_MODEL_DIR.exists()):
    trainer.train()
    trainer.save_model(str(BEST_MODEL_DIR))
    tokenizer.save_pretrained(str(BEST_MODEL_DIR))
    print('Saved best model to', BEST_MODEL_DIR)
else:
    print('Skip training. Using existing model at', BEST_MODEL_DIR)


In [ ]:
pred_out = trainer.predict(test_tok)
test_metrics = pred_out.metrics

pred_ids = np.argmax(pred_out.predictions, axis=-1)
true_ids = pred_out.label_ids

y_true = []
y_pred = []
for pseq, lseq in zip(pred_ids, true_ids):
    for p, l in zip(pseq, lseq):
        if l == -100:
            continue
        y_true.append(id2label[int(l)])
        y_pred.append(id2label[int(p)])

critical = ['ADV', 'PART', 'NOUN', 'VERB']
crit_indices = [i for i, t in enumerate(y_true) if t in critical]
crit_acc = sum(1 for i in crit_indices if y_true[i] == y_pred[i]) / max(len(crit_indices), 1)

test_summary = {
    'model_name': MODEL_NAME,
    'eval_loss': float(test_metrics.get('test_loss', 0.0)),
    'precision': float(test_metrics.get('test_precision', 0.0)),
    'recall': float(test_metrics.get('test_recall', 0.0)),
    'f1': float(test_metrics.get('test_f1', 0.0)),
    'critical_token_accuracy': float(crit_acc),
    'num_test_tokens': len(y_true),
}

print(test_summary)
with (OUT_DIR / 'indobert_test_metrics.json').open('w', encoding='utf-8') as f:
    json.dump(test_summary, f, ensure_ascii=False, indent=2)
print('Saved metrics:', OUT_DIR / 'indobert_test_metrics.json')


In [ ]:
TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def tokenize_text(text: str) -> List[str]:
    return TOKEN_RE.findall(text)

def predict_tokens(tokens: List[str]) -> List[str]:
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    enc = {k: v.to(trainer.model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = trainer.model(**enc).logits[0].cpu().numpy()
    pred_ids = np.argmax(logits, axis=-1)
    word_ids = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    out = []
    seen = set()
    for tidx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        out.append(id2label[int(pred_ids[tidx])])
    if len(out) < len(tokens):
        out += ['NOUN'] * (len(tokens) - len(out))
    return out[:len(tokens)]

pantun_df = pd.read_csv(PANTUN_CSV)
rows = []
limit = 100

for pantun_index, row in pantun_df.head(limit).iterrows():
    text = str(row.get('text_pantun', '')).replace('\\n', '\n')
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for line_no, line_text in enumerate(lines, start=1):
        tokens = tokenize_text(line_text)
        if not tokens:
            continue
        pred_tags = predict_tokens(tokens)
        token_tag_pairs = ' | '.join(f'{t}/{g}' for t, g in zip(tokens, pred_tags))
        rows.append({
            'id': row.get('id', ''),
            'pantun_index': int(pantun_index) + 1,
            'line_no': line_no,
            'line_text': line_text,
            'token_count': len(tokens),
            'tokens': ' '.join(tokens),
            'pred_tags': ' '.join(pred_tags),
            'token_tag_pairs': token_tag_pairs,
        })

pred_df = pd.DataFrame(rows)
pred_out_csv = OUT_DIR / 'pantun_gaul_pred_100_indobert.csv'
pred_df.to_csv(pred_out_csv, index=False, encoding='utf-8')

print('[OK] output:', pred_out_csv)
print('[INFO] pantun processed:', min(limit, len(pantun_df)))
print('[INFO] line predictions:', len(pred_df))
display(pred_df.head(10))


In [ ]:
mini_gold_path = DATA_DIR / 'processed_crf' / 'mini_gold_template_400.csv'
if not mini_gold_path.exists():
    print('Mini-gold tidak ditemukan:', mini_gold_path)
else:
    mg = pd.read_csv(mini_gold_path)
    pred = pd.read_csv(OUT_DIR / 'pantun_gaul_pred_100_indobert.csv')

    token_rows = []
    for _, row in pred.iterrows():
        toks = str(row['tokens']).split()
        tags = str(row['pred_tags']).split()
        for i, (tok, tag) in enumerate(zip(toks, tags), start=1):
            token_rows.append({
                'pantun_index': int(row['pantun_index']),
                'line_no': int(row['line_no']),
                'token_index': i,
                'token': tok,
                'pred_tag_indobert': tag,
            })
    pred_tok = pd.DataFrame(token_rows)

    key_cols = ['pantun_index', 'line_no', 'token_index', 'token']
    merged = mg.merge(pred_tok, on=key_cols, how='left')
    merged['pred_tag'] = merged['pred_tag_indobert'].fillna(merged['pred_tag'])

    eval_csv = OUT_DIR / 'mini_gold_with_indobert_pred.csv'
    merged.to_csv(eval_csv, index=False, encoding='utf-8')

    !python scripts/article5/evaluate_mini_gold_pos.py \
      --gold-csv {eval_csv} \
      --out-metrics-json {OUT_DIR / 'mini_gold_metrics_indobert.json'} \
      --out-confusion-csv {OUT_DIR / 'mini_gold_confusion_indobert.csv'}

    print('Saved:', OUT_DIR / 'mini_gold_metrics_indobert.json')
    print('Saved:', OUT_DIR / 'mini_gold_confusion_indobert.csv')
